# 법률 문서 임베딩 학습 및 평가

한국어 Sentence-BERT 모델을 이용해 판례 검색 기준선을 만들고, 유사 문장 쌍으로 선택적 미세조정을 수행함. 현재 포함된 데이터는 기능 검증용 합성 데이터이므로 최종 실험에서는 출처가 명확한 실제 판례로 교체함.

In [ ]:
from pathlib import Path
from time import perf_counter
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.sentence_transformer.training_args import BatchSamplers

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import documents_from_cases, load_cases

PROCESSED_DIR = ROOT / 'data' / 'processed'
DATA_PATH = PROCESSED_DIR / 'cases.csv'
PAIR_PATH = PROCESSED_DIR / 'training_pairs.csv'
VAL_DATA_PATH = PROCESSED_DIR / 'validation_cases.csv'
VAL_PAIR_PATH = PROCESSED_DIR / 'validation_pairs.csv'
if not DATA_PATH.exists():
    DATA_PATH = ROOT / 'data' / 'sample_cases.csv'
    PAIR_PATH = ROOT / 'data' / 'training_pairs.csv'
    VAL_DATA_PATH = DATA_PATH
    VAL_PAIR_PATH = PAIR_PATH
EPOCH1_MODEL_PATH = ROOT / 'models' / 'legal-sbert'
EPOCH3_MODEL_PATH = ROOT / 'models' / 'legal-sbert-epoch3'
SELECTED_MODEL_FILE = ROOT / 'models' / 'selected_model.txt'
BASE_MODEL = 'jhgan/ko-sroberta-multitask'
MINILM_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 1. 데이터 확인 및 전처리

In [ ]:
cases = load_cases(DATA_PATH)
pairs = pd.read_csv(PAIR_PATH)
print(f'판례: {len(cases):,}건 / 학습 쌍: {len(pairs):,}개')
print('실제 AI Hub 데이터 사용:', 'processed' in str(DATA_PATH))
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
category_counts = cases['category'].value_counts().head(15).sort_values()
category_counts.plot.barh(title='사건 분야 분포')
plt.xlabel('판례 수')
plt.tight_layout()
plt.show()

In [ ]:
documents = documents_from_cases(cases)
print(f'임베딩 대상 문서: {len(documents):,}건')
print('개별 판례 원문은 Notebook 출력에 표시하지 않습니다.')

## 2. 사전학습 모델 성능 비교

동일한 검증 데이터에서 다국어 MiniLM과 한국어 Ko-SRoBERTa의 Recall@K와 처리 시간을 비교함

In [ ]:
validation_cases = load_cases(VAL_DATA_PATH)
validation_pairs = pd.read_csv(VAL_PAIR_PATH)

if 'case_id' in validation_pairs.columns:
    validation_pairs['case_id'] = validation_pairs['case_id'].astype(str)
    valid_ids = set(validation_cases['id'].astype(str))
    eval_pairs = validation_pairs[validation_pairs['case_id'].isin(valid_ids)].head(500)
else:
    validation_cases = validation_cases.head(500)
    eval_pairs = pd.DataFrame({
        'anchor': validation_cases['issues'],
        'case_id': validation_cases['id'].astype(str),
    })

eval_documents = documents_from_cases(validation_cases)
case_index = {
    str(case_id): index for index, case_id in enumerate(validation_cases['id'])
}
target_indices = np.array([case_index[case_id] for case_id in eval_pairs['case_id']])

def evaluate_retrieval(target_model):
    corpus_embeddings = target_model.encode(
        eval_documents, convert_to_numpy=True, normalize_embeddings=True,
        show_progress_bar=True,
    )
    query_embeddings = target_model.encode(
        eval_pairs['anchor'].tolist(), convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=True,
    )
    similarities = query_embeddings @ corpus_embeddings.T
    metrics = {}
    for k in [1, 3, 5]:
        top_indices = np.argsort(similarities, axis=1)[:, ::-1][:, :k]
        metrics[f'Recall@{k}'] = np.mean([
            target in candidates
            for target, candidates in zip(target_indices, top_indices)
        ])
    return metrics

benchmark_models = {
    'Multilingual MiniLM': MINILM_MODEL,
    'Ko-SRoBERTa': BASE_MODEL,
}
benchmark_rows = []
model = None

for model_label, model_name in benchmark_models.items():
    candidate_model = SentenceTransformer(model_name)
    started_at = perf_counter()
    metrics = evaluate_retrieval(candidate_model)
    elapsed_seconds = perf_counter() - started_at
    benchmark_rows.append({
        'model': model_label,
        **metrics,
        'seconds': elapsed_seconds,
    })
    print(model_label, metrics, f'{elapsed_seconds:.1f}초')
    if model_label == 'Ko-SRoBERTa':
        model = candidate_model

baseline_metrics = next(
    {key: value for key, value in row.items() if key.startswith('Recall@')}
    for row in benchmark_rows if row['model'] == 'Ko-SRoBERTa'
)

## 3. 법률 유사 문장 쌍 미세조정

`RUN_TRAINING=True`이면 기본 Ko-SRoBERTa에서 3 Epoch 미세조정을 시작함. 기존 1 Epoch 모델은 보존하고 새 모델은 `models/legal-sbert-epoch3`에 별도 저장함.

In [ ]:
RUN_TRAINING = True  # 직접 학습할 때 True로 변경
EPOCHS = 3
BATCH_SIZE = 8
MAX_TRAIN_PAIRS = 5000  # 첫 실행 성공 후 20000으로 확대

if RUN_TRAINING:
    # 셀을 다시 실행해도 기존 학습 모델에서 이어서 학습하지 않도록 초기화
    model = SentenceTransformer(BASE_MODEL)
    selected_pairs = pairs.sample(
        n=min(MAX_TRAIN_PAIRS, len(pairs)), random_state=42
    )
    train_dataset = Dataset.from_pandas(
        selected_pairs[['anchor', 'positive']], preserve_index=False
    )
    train_loss = MultipleNegativesRankingLoss(model)
    training_args = SentenceTransformerTrainingArguments(
        output_dir=str(ROOT / 'models' / 'checkpoints-epoch3'),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        learning_rate=2e-5,
        warmup_steps=0.1,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        fp16=torch.cuda.is_available(),
        bf16=False,
        logging_steps=10,
        save_strategy='no',
        report_to=[],
        dataloader_num_workers=0,
    )
    trainer = SentenceTransformerTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        loss=train_loss,
    )
    trainer.train()
    model.save_pretrained(str(EPOCH3_MODEL_PATH))

    loss_history = pd.DataFrame([
        item for item in trainer.state.log_history if 'loss' in item
    ])
    if not loss_history.empty:
        loss_history.plot(x='step', y='loss', title='파인튜닝 학습 Loss')
        plt.ylabel('Loss')
        plt.grid(alpha=0.3)
        plt.show()
    print(f'학습 완료: {len(selected_pairs):,}쌍')
    print('3 Epoch 모델 저장:', EPOCH3_MODEL_PATH)
else:
    print('학습을 실행하려면 RUN_TRAINING을 True로 변경하세요.')

## 4. 최종 모델 평가 및 검색 예시

In [ ]:
experiment_paths = {
    'Legal Ko-SRoBERTa (1 Epoch)': EPOCH1_MODEL_PATH,
    'Legal Ko-SRoBERTa (3 Epoch)': EPOCH3_MODEL_PATH,
}
comparison_rows = benchmark_rows.copy()
trained_models = {}

for model_label, model_path in experiment_paths.items():
    if not model_path.exists():
        print(f'모델 없음 - 비교 제외: {model_path}')
        continue
    candidate_model = SentenceTransformer(str(model_path))
    started_at = perf_counter()
    metrics = evaluate_retrieval(candidate_model)
    elapsed_seconds = perf_counter() - started_at
    comparison_rows.append({
        'model': model_label,
        **metrics,
        'seconds': elapsed_seconds,
    })
    trained_models[model_label] = candidate_model

if not trained_models:
    raise FileNotFoundError('비교할 학습 모델이 없습니다. 먼저 학습 셀을 실행하세요.')

comparison = pd.DataFrame(comparison_rows).set_index('model')
display(comparison.round(4))

trained_labels = list(trained_models)
best_model_label = comparison.loc[trained_labels, 'Recall@5'].idxmax()
final_model = trained_models[best_model_label]
selected_model_path = experiment_paths[best_model_label]
SELECTED_MODEL_FILE.write_text(str(selected_model_path.resolve()), encoding='utf-8')
print('Recall@5 기준 최종 모델:', best_model_label)
print('Streamlit 선택 모델:', selected_model_path)

comparison[['Recall@1', 'Recall@3', 'Recall@5']].plot.bar(
    figsize=(10, 5), title='임베딩 모델 검색 성능 비교', rot=0
)
plt.ylabel('Recall')
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

comparison['seconds'].plot.bar(
    figsize=(8, 4), title='모델별 검증 데이터 처리 시간', rot=0,
    color=['#4C78A8', '#F58518', '#54A24B', '#E45756'][:len(comparison)],
)
plt.ylabel('초')
plt.tight_layout()
plt.show()

## 5. Streamlit용 전체 인덱스 저장

1 Epoch와 3 Epoch 중 Recall@5가 높은 모델로 앱에서 사용할 임베딩과 판례 메타데이터를 갱신함. Streamlit을 다시 시작하면 `selected_model.txt`에 기록된 모델을 사용함.

In [ ]:
final_document_embeddings = final_model.encode(
    documents, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True,
)

query = '인터넷에서 산 물건이 고장 났는데 판매자가 환불을 해주지 않는다'
query_vector = final_model.encode(
    [query], convert_to_numpy=True, normalize_embeddings=True
)[0]
scores = final_document_embeddings @ query_vector
top_indices = np.argsort(scores)[::-1][:3]
result = cases.iloc[top_indices][['case_name', 'category', 'issues']].copy()
result.insert(0, 'similarity', scores[top_indices])
display(result)

artifact_dir = ROOT / 'artifacts'
artifact_dir.mkdir(exist_ok=True)
np.save(artifact_dir / 'case_embeddings.npy', final_document_embeddings.astype('float32'))
cases.to_csv(artifact_dir / 'indexed_cases.csv', index=False, encoding='utf-8-sig')
print('인덱스 저장 완료:', artifact_dir)